# Signatures comparison — new sorted-cell held-out cohort (Figure 4)

The new sorted-cell cohort serves as a true held-out **test** cohort for the cell-type FGES benchmark, answering Reviewer 1's request to "Add new data of sorted cells to cell signature comparison" and the paper Methods commitment to ~75/25 train/test separation.

**Scope:** 16 of 20 FGES. The four rare-GOI FGES — `Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`, `Main4_Plasma_cells` — have too few held-out samples; in the supplementary heatmaps they are backfilled from cross-validated train-cohort scores.

**Random-FGES baseline:** v1 random gene lists are reused (loaded from `msigdb_gmt_paperS1.pkl`) but rescored on the new cohort so ranks stay comparable.

**Data:** input data is not distributed with this repository. Set the `<PATH_TO_...>` placeholders in the paths cell below before running. Outputs (pickles, SVGs, tables) go to `OUTPUT_DIR` with a `_new_cohort` suffix.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from loguru import logger

from signature_validation.benchmark.cohorts import (
    CONTROLS_ORDER,
    EXCLUDED_FGES_RARE,
    MAP_RAW,
    build_mapping,
    intersect_controls_with_cohort,
    load_new_cohort_annotation,
    load_new_cohort_expressions,
)
from signature_validation.benchmark.plotting import (
    plot_sens_spec_scatter,
    plot_signature_heatmap,
    plot_violin_per_source,
)
from signature_validation.benchmark.scoring import (
    compute_mapping_ssgseas,
    compute_out_table,
    fdr_correct_out,
)
from signature_validation.benchmark.signatures import (
    count_random_fges,
    harmonize_gmt_to_index,
    load_v1_msigdb_gmt,
    select_msigdb_gmt_subset,
)
from signature_validation.plotting.plotting import cells_p

sns.set_style("white")
plt.rcParams["svg.fonttype"] = "none"

In [ ]:
# ── Пути к данным (данные не распространяются с репозиторием) ─────────────
# Заполнить плейсхолдеры перед запуском.
NEW_ANNOT_PATH = Path("<PATH_TO_NEW_COHORT_ANNOTATION_TSV>")          # аннотация новой (held-out) когорты
EXPR_PATH = "<PATH_TO_SORTED_CELL_EXPRESSIONS_DIR>"                  # экспрессии, только для RECOMPUTE=True
TRAIN_SSGSEAS_PATH = Path("<PATH_TO_TRAIN_MAPPING_SSGSEAS_PKL>")      # mapping_ssgseas опубликованной (train) когорты
TRAIN_ANNOT_PATH = Path("<PATH_TO_TRAIN_CELLS_ANNOTATION_TSV>")       # аннотация train-когорты (для S6.1)
CROSSVAL_SSGSEAS_PATH = "<PATH_TO_CROSSVAL_MAPPING_SSGSEAS_PKL>"      # кроссвалидационные скоры train (бэкфилл)

OUTPUT_DIR = Path("../plots/new_cohort/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAPPING_SSGSEAS_PATH = OUTPUT_DIR / "mapping_ssgseas_new_cohort.pkl"
FGES_METRICS_PATH = OUTPUT_DIR / "fges_metrics_new_cohort.pkl"
OUT_TSV_PATH = OUTPUT_DIR / "out_new_cohort.tsv"
HEATMAP_PATH = OUTPUT_DIR / "signature_heatmap_new_cohort.svg"

# ── Режим работы ──────────────────────────────────────────────────────────
# RECOMPUTE=True  — пересчитать ssGSEA (mapping_ssgseas) и fges_metrics с нуля
#                   (медленно; нужны экспрессии EXPR_PATH), затем сохранить pickle.
# RECOMPUTE=False — загрузить готовые pickle (MAPPING_SSGSEAS_PATH /
#                   FGES_METRICS_PATH) и только строить графики/таблицы
#                   (экспрессии не грузятся).
RECOMPUTE = True

logger.info("RECOMPUTE={}", RECOMPUTE)
logger.info("new annotation:    {}", NEW_ANNOT_PATH)
logger.info("mapping_ssgseas:   {}", MAPPING_SSGSEAS_PATH)

In [ ]:
public_cells_annot = load_new_cohort_annotation(NEW_ANNOT_PATH)
public_cells_annot["Cell_type"].value_counts()

In [ ]:
# ── Held-out тест: убираем сэмплы, которые уже были в опубликованной когорте ──
# 65 из 30 411 сэмплов новой когорты встречаются в v1-пикле (train). Пока они
# здесь, Figure 4 частично меряет те же данные, на которых сигнатуры отбирались.
# Удаление концентрированное: страдают ровно два типа — Mast_cells 58 → 8
# (уходит под порог min_n=20 и попадёт в merge-бэкфилл, см. ячейку подготовки
# доп. фигур) и Follicular_T_helpers 265 → 250 (без последствий).
from signature_validation.benchmark.cohorts import (
    drop_train_samples,
    load_train_sample_ids,
)

train_ids = load_train_sample_ids(TRAIN_SSGSEAS_PATH)
public_cells_annot, train_drop_report = drop_train_samples(public_cells_annot, train_ids)
train_drop_report

In [ ]:
from signature_validation.utils.utils import read_expressions

In [ ]:
# Экспрессии нужны только для пересчёта (ssGSEA + метрики).
if RECOMPUTE:
    public_cells_expr = read_expressions(public_cells_annot, path=EXPR_PATH)
    logger.info("expressions: {}", public_cells_expr.shape)
else:
    public_cells_expr = None
    logger.info("RECOMPUTE=False — загрузку экспрессий пропускаю")

In [ ]:
V1_GMT_PICKLE = "./data/msigdb_gmt_paperS1.pkl"
v1_gmt_full = load_v1_msigdb_gmt(V1_GMT_PICKLE)
in_scope_fges = [k for k in MAP_RAW if k not in EXCLUDED_FGES_RARE]
v1_gmt = select_msigdb_gmt_subset(v1_gmt_full, in_scope_fges)

for sign in in_scope_fges:
    assert sign in v1_gmt[sign], f"v1 GMT[{sign}] missing the BG sub-signature"
    n_random = count_random_fges(v1_gmt[sign])
    assert n_random == 10, f"{sign}: expected 10 RANDOM_FGES, got {n_random}"

if RECOMPUTE:
    msigdb_gmt = harmonize_gmt_to_index(v1_gmt, public_cells_expr.index)
else:
    # Без пересчёта нужны только ИМЕНА суб-сигнатур (не гены) — берём v1_gmt как есть.
    msigdb_gmt = v1_gmt
logger.info(
    "msigdb_gmt: {} FGES, {} signatures total",
    len(msigdb_gmt),
    sum(len(v) for v in msigdb_gmt.values()),
)

In [ ]:
mapping = build_mapping(annotation=public_cells_annot)
controls_present = intersect_controls_with_cohort(CONTROLS_ORDER, public_cells_annot)
logger.info(
    "in-scope: {} FGES; controls present in new cohort: {}",
    len(mapping),
    len(controls_present),
)
for sign, bucket in mapping.items():
    logger.info(
        "{}: GOI={}, Control={}, Deleted={}",
        sign,
        bucket["Goi"],
        len(bucket["Control"]),
        len(bucket["Deleted_controls"]),
    )

In [ ]:
from signature_validation.benchmark.cohorts import PARENT_TO_DAUGHTER
from signature_validation.benchmark.scoring import clean_parent_daughter_goi

if RECOMPUTE:
    mapping_ssgseas = compute_mapping_ssgseas(
        public_cells_expr=public_cells_expr,
        public_cells_annot=public_cells_annot,
        mapping=mapping,
        msigdb_gmt=msigdb_gmt,
    )
    # Очистка parent→daughter (Scater_plots ячейка 16): без неё скоры
    # Macrophages/Monocyte искажены (общие сигнатуры остаются в GOI родителя).
    mapping_ssgseas = clean_parent_daughter_goi(mapping_ssgseas, PARENT_TO_DAUGHTER)
    with open(MAPPING_SSGSEAS_PATH, "wb") as fh:
        pickle.dump(mapping_ssgseas, fh, pickle.HIGHEST_PROTOCOL)
    logger.info("computed+cleaned mapping_ssgseas → {}", MAPPING_SSGSEAS_PATH)
else:
    with open(MAPPING_SSGSEAS_PATH, "rb") as fh:
        mapping_ssgseas = pickle.load(fh)
    logger.info("loaded mapping_ssgseas from {}", MAPPING_SSGSEAS_PATH)

for sign in mapping_ssgseas:
    if sign in EXCLUDED_FGES_RARE:
        continue
    assert mapping_ssgseas[sign]["Goi"], f"{sign}: GOI cohort is empty"

## Score correctness, metrics, scatters and Supplement tables

`clean_parent_daughter_goi` reproduces the v1 `parent_to_daughter` cleaning (Scater_plots cell 16) that the first draft omitted — without it macrophage/monocyte GOI frames keep daughter-shared signatures and the heatmap / `out` table / scatter are wrong. Then `fges_metrics` (bootstrap F1/AUC + rank CV), the F1-vs-CV source scatters (Fig 4 F/G) and the S4/S6 Supplement tables.

In [ ]:
# Проверка счётчиков образцов для макрофагов — только в режиме пересчёта
# (нужны экспрессии): аннотация vs. экспрессии vs. GOI-фреймы.
if RECOMPUTE:
    MACRO_GOI = {
        "Main4_Pan_macrophage_signature": "Macrophages",
        "Main4_M2_signature": "Macrophages_M2",
        "Main4_Monocyte": "Monocytes",
    }

    expr_cols = set(public_cells_expr.columns)
    ct_counts = public_cells_annot["Cell_type"].value_counts()

    for fges, ct in MACRO_GOI.items():
        n_annot = int(ct_counts.get(ct, 0))
        samples_of_ct = public_cells_annot.index[public_cells_annot["Cell_type"] == ct]
        n_expr = len(expr_cols.intersection(samples_of_ct))
        goi_frames = mapping_ssgseas.get(fges, {}).get("Goi", {})
        n_goi = int(goi_frames[ct].shape[0]) if ct in goi_frames else 0
        dropped = n_annot - n_expr
        logger.info(
            "{ct:16s} | FGES={fges:34s} | annot={n_annot:5d} | with_expr={n_expr:5d} "
            "| GOI_frame_rows={n_goi:5d} | dropped_no_expr={dropped:5d}",
            ct=ct, fges=fges, n_annot=n_annot, n_expr=n_expr, n_goi=n_goi, dropped=dropped,
        )
        if n_goi != n_expr:
            logger.warning(
                "{ct}: GOI-фрейм ({n_goi}) != образцов с экспрессиями ({n_expr}) — "
                "проверь дубликаты индексов / фильтрацию",
                ct=ct, n_goi=n_goi, n_expr=n_expr,
            )

    all_cts = {ct for b in mapping.values() for g in ("Goi", "Control", "Deleted_controls") for ct in b[g]}
    total_samples = public_cells_annot.index[public_cells_annot["Cell_type"].isin(all_cts)]
    total_with_expr = len(expr_cols.intersection(total_samples))
    logger.info("ОБЩЕЕ образцов (все in-scope cell types, с экспрессиями): {}", total_with_expr)
else:
    logger.info("RECOMPUTE=False — проверку счётчиков макрофагов пропускаю")

In [ ]:
out = compute_out_table(mapping_ssgseas, mapping, msigdb_gmt, controls_present)
out = fdr_correct_out(out, controls_present)
out.to_csv(OUT_TSV_PATH, sep="\t")
logger.info("wrote {} ({} rows × {} cols)", OUT_TSV_PATH, *out.shape)
out.head()

In [ ]:
# plot_violin_per_source(mapping_ssgseas, save_dir=OUTPUT_DIR)
plot_signature_heatmap(
    mapping_ssgseas=mapping_ssgseas,
    out_df=out,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
    annotation=public_cells_annot,
    controls_order=controls_present,
    palette={ct: cells_p[ct] for ct in controls_present if ct in cells_p},
    save_path=HEATMAP_PATH,
    short=True,
)
averaged = plot_sens_spec_scatter(
    mapping_ssgseas=mapping_ssgseas,
    msigdb_gmt=msigdb_gmt,
    mapping=mapping,
    save_dir=OUTPUT_DIR,
)
logger.info("plots saved under {}", OUTPUT_DIR)

In [ ]:
msigdb_gmt['Main4_Pan_macrophage_signature']['Main4_Pan_macrophage_signature'].genes

In [ ]:
# fges_metrics: bootstrap-классификация (F1/Accuracy/PR-AUC/ROC-AUC) + CV рангов
# (порт Scater_plots ячейки 20). Пересчёт или загрузка готового pickle.
from signature_validation.benchmark.metrics import compute_fges_metrics

# get_strat_cell_type берёт min_samples = МИНИМУМ по контрольным типам, поэтому
# один крошечный контроль обрезает все подвыборки. В референсе (Scater_plots
# ячейка 19) для этого дропали только Th2_cells, но в новой когорте узкое место
# шире: Monocytic_DC=1, Plasmablasts=3, Th2_cells=8, Th1_cells=8 семплов со
# скорами → min_samples=1 и контроль схлопнулся бы до ~37 семплов на итерацию.
# Дропаем все контроли ниже порога; на GOI-когорты это не влияет — они берутся
# из mapping_ssgseas["Goi"], а public_cells_annot нужен только для стратификации
# контролей (т.е. Th1_cells остаётся полноценным GOI для Main4_Th1_signature).
MIN_CONTROL_SAMPLES = 10

if RECOMPUTE:
    ranked_expr = public_cells_expr.rank(pct=True)
    pipeline_genes = public_cells_expr.index.to_list()

    control_samples = pd.Index(
        sorted(
            {
                sample
                for bucket in mapping_ssgseas.values()
                for df in bucket.get("Control", {}).values()
                for sample in df.index
            }
        )
    )
    control_counts = (
        public_cells_annot["Cell_type"].reindex(control_samples).dropna().value_counts()
    )
    tiny_controls = sorted(control_counts.index[control_counts < MIN_CONTROL_SAMPLES])
    logger.info(
        "дроп контролей < {} семплов: {} | min_samples станет {}",
        MIN_CONTROL_SAMPLES,
        {ct: int(control_counts[ct]) for ct in tiny_controls},
        int(control_counts[control_counts >= MIN_CONTROL_SAMPLES].min()),
    )
    annot_for_metrics = public_cells_annot[
        ~public_cells_annot["Cell_type"].isin(tiny_controls)
    ]
    

    fges_metrics = compute_fges_metrics(
        mapping_ssgseas=mapping_ssgseas,
        msigdb_gmt=msigdb_gmt,
        public_cells_annot=annot_for_metrics,
        ranked_expr=ranked_expr,
        pipeline_genes=pipeline_genes,
        n_iter=10,
    )
    with open(FGES_METRICS_PATH, "wb") as fh:
        pickle.dump(fges_metrics, fh, pickle.HIGHEST_PROTOCOL)
    logger.info("computed fges_metrics → {} ({} FGES cols)", FGES_METRICS_PATH, len(fges_metrics))
else:
    with open(FGES_METRICS_PATH, "rb") as fh:
        fges_metrics = pickle.load(fh)
    logger.info("loaded fges_metrics from {} ({} FGES cols)", FGES_METRICS_PATH, len(fges_metrics))

In [ ]:
# Скаттеры F1 vs CV рангов, по источникам FGES (как Scater_plots панели F/G).
# Пишем в OUTPUT_DIR (hub scratch), чтобы НЕ перезаписать опубликованные paper-SVG.
from signature_validation.benchmark.metrics import plot_f1_cv_scatters

plot_f1_cv_scatters(
    fges_metrics=fges_metrics,
    msigdb_gmt=msigdb_gmt,
    save_dir=OUTPUT_DIR,
)
logger.info("F1/CV scatters → {}", OUTPUT_DIR / "svg_pictures_F1_cv")

In [ ]:
# Supplementary таблицы: S4.x (перформанс FGES на клетку, как пример B_cells)
# + S6.1 (инвентарь датасетов: сколько сэмплов каждого датасета реально попало
# в фигуры и как они делятся на train / holdout).
import os

from signature_validation.benchmark.cohorts import (
    collect_sample_ids,
    load_train_annotation,
)
from signature_validation.benchmark.tables import (
    build_dataset_list_table,
    build_fges_performance_tables,
)

TABLES_DIR = (OUTPUT_DIR / "tables").resolve()
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# detect_fges_source читает ./data/msigdb...gmt относительно cwd — временно
# переходим на уровень выше (Cell_type_FGES_comparison), где эта папка есть.
_cwd0 = os.getcwd()
try:
    os.chdir("..")
    s4_tables = build_fges_performance_tables(
        mapping_ssgseas=mapping_ssgseas,
        fges_metrics=fges_metrics,
        mapping=mapping,
        msigdb_gmt=msigdb_gmt,
        save_dir=TABLES_DIR,
        prefix="S4",
    )
finally:
    os.chdir(_cwd0)

# ── S6.1: Sorted cell dataset list ───────────────────────────────────────
# N считается по ПРОСКОРЕННЫМ сэмплам (union Goi + Control + Deleted_controls),
# а не по строкам аннотации: скор есть только у сэмпла, который прошёл QC и имел
# экспрессии, то есть ровно у того, что видно на фигурах. По аннотации вышло бы
# 30 411 сэмплов вместо 16 976 — таблица описывала бы не тот эксперимент.
#
# Train — вся опубликованная v1-когорта (7 990 сэмплов): именно на ней отбирались
# сигнатуры, и именно из неё кроссвалидационный бэкфилл добирает редкие типы
# клеток (Th17 / Eosinophils / Endothelium_lymph) для supp-фигур.
#
# fallback_annotation читается ЗАНОВО, а не берётся из public_cells_annot: 44
# train-сэмпла (tonsil Tfh и mast cells, добавленные в v1 вручную)
# отсутствуют в v1-аннотации и есть только в новой — но их же и удалил
# drop_train_samples, поэтому post-drop переменная их уже не содержит.
holdout_ids = collect_sample_ids(mapping_ssgseas)
train_annot = load_train_annotation(
    train_ids,
    path=TRAIN_ANNOT_PATH,
    fallback_annotation=load_new_cohort_annotation(NEW_ANNOT_PATH),
)

s6_table = build_dataset_list_table(
    public_cells_annot,
    TABLES_DIR / "S6.1_sorted_cell_datasets.tsv",
    holdout_samples=holdout_ids,
    train_samples=train_ids,
    train_annotation=train_annot,
)

assert s6_table["N_holdout"].sum() == len(holdout_ids)
assert s6_table["N_train"].sum() == len(train_ids)
assert (s6_table["N_samples"] == s6_table["N_train"] + s6_table["N_holdout"]).all()
assert s6_table["Dataset"].notna().all()
assert not s6_table["Dataset"].duplicated().any()

_both = int(((s6_table["N_train"] > 0) & (s6_table["N_holdout"] > 0)).sum())
logger.info(
    "wrote {} S4 tables + S6.1: {} датасетов | train {} | holdout {} | "
    "в обеих когортах {} → {}",
    len(s4_tables),
    len(s6_table),
    int(s6_table["N_train"].sum()),
    int(s6_table["N_holdout"].sum()),
    _both,
    TABLES_DIR,
)
s6_table.head()

### Table S6.1 — что писать в подписи

**Caption (paper-ready).** *Table S6.1. Sorted cell dataset list.* Datasets contributing
samples to the cell-type FGES benchmark. `N_samples` counts only samples with a computed
ssGSEA score — i.e. samples that passed technical QC, carried expression data and belong to
one of the benchmarked cell types — split into the published training cohort (`N_train`,
7,990 samples) and the held-out sorted-cell cohort (`N_holdout`, 16,976 samples).
`Cell_types` lists the benchmarked cell types the dataset contributes, semicolon-separated.

**Оговорки, которые стоит держать в тексте методов:**

- **Знаменатель.** 1 300 датасетов / 24 966 сэмплов — это уровень *проскоренных* сэмплов.
  Если считать по строкам аннотации, будет 1 259 (new) + 736 (old) датасетов и 30 411
  сэмплов только в holdout: другое число, другой смысл. Любые «shared studies» надо
  цитировать с указанием знаменателя.
- **Сплит на уровне сэмплов, а не исследований.** 128 датасетов дают сэмплы в обе когорты
  (468 только train, 704 только holdout). Пересечение по сэмплам при этом ровно нулевое —
  `drop_train_samples` убирает 65 совпадающих ID. То есть holdout честно held-out
  по сэмплам, но не по батчам.
- **Редкие типы клеток идут из train.** `Th17_cells`, `Eosinophils`, `Endothelium_lymph`
  не имеют holdout-сэмплов вообще, а `Th2_cells` / `Mast_cells` / `Monocytic_DC` имеют
  меньше 10; в supplementary-фигурах они добираются кроссвалидацией train-когорты
  (`backfill_rare_cell_types`, `min_n=10`, `mode="merge"`). Эти сэмплы уже посчитаны
  в колонке `N_train`.
- **Лейблы гармонизированы.** Tonsillar Tfh из train (`Follicular_T_helper_tonsil`) сведён
  к `Follicular_T_helpers`, иначе одна популяция была бы перечислена дважды. Биологическая
  эквивалентность двух определений из аннотации не следует — если это важно рецензенту,
  надо смотреть на маркеры сортировки.
- **Внутренний датасет** — один датасет в таблице внутренний, публичного accession у него нет.

## Доп. фигуры — медианы ssGSEA по типам клеток

Supplementary-фигуры поверх того же `mapping_ssgseas`, все на **медианах сырого (нешкалированного) ssGSEA** по типам клеток.

**Ось X (все фигуры), строго в этом порядке:** T cells, CD4 T cells, Th1 cells, Th2 cells, Th17 cells, Follicular T helpers, Tregs, CD8 T cells, NK cells, B cells, Plasma B cells, Plasmablasts, Neutrophils, Eosinophils, Mast cells, Monocytes, Macrophages, Macrophages M1, Macrophages M2, Endothelium, Endothelium lymph, **Other controls** — в последнюю колонку пулятся все прочие проскоренные типы.

**Оформление (общее для обоих наборов).** Ячейки **квадратные**; цифр внутри ячеек **нет** — их значение несёт colorbar. Цвет кодирует **сырую медиану ssGSEA** (не z-score), поэтому colorbar подписан в единицах ssGSEA и читается напрямую. Пределы цвета берутся по **полной** матрице всех FGES и только потом нарезаются по файлам, иначе цвета двух файлов несопоставимы.

Шкала цвета линейная. `color_scale="symlog"` доступен (знаковый log10: линейный внутри ±`linthresh`, log10 снаружи — обычный log10 не годится, ~8% медиан отрицательные), но по умолчанию не включён: медианы занимают меньше одной декады (99.5% ячеек в 10³–10⁴), поэтому log10 сжимает полезный контраст (Macrophages 8688 против NK 2955 → 3.94 против 3.47) в одну двенадцатую полосы, а остальное тратит на почти пустую область вокруг нуля.

**Набор 1 — по одному SVG на FGES, на каждый способ отбора.** Пишется в `heatmaps_per_fges/<отбор>/<FGES>.svg` плюс `_colorbar.svg` на папку.

Основной отбор — `separation`, «**наиболее высокий в GOI, наиболее низкий в контроле**». Строки ранжируются по **разрыву медиан**:

```
gap = median(GOI) − median(Control)
```

Ранжирование именно по `gap` потому, что это ровно то разделение, которое читатель видит на фигуре: ячейки покрашены сырыми медианами, значит самый широкий разрыв — это самый широкий скачок цвета. Cohen's d `(mean(GOI) − mean(Control)) / pooled_sd` считается рядом и попадает в таблицу, но не участвует в сортировке: он делит на объединённое SD и потому опускает сигнатуру за широкий разброс внутри GOI, которого на фигуре не видно. Обратная сторона `gap`: суб-сигнатуры одной FGES живут на разных шкалах ssGSEA, поэтому крупный по абсолютной величине gene set может выиграть по разрыву, не разделяя когорты лучше в относительных единицах — для этого в таблице и лежит колонка `d`.

Считается прямо по тому ssGSEA, который фигура и рисует, а не по `fges_metrics`. Отсюда два следствия: кроссвалидационный бэкфилл учтён (`fges_metrics` про него не знает), и один и тот же MSigDb-сет под двумя FGES скорится отдельно под каждой — то есть исчезает режим `ambiguous_metric`, когда сет наследовал GOI/control-разбиение чужого блока. `Deleted_controls` в контроль **не** входят: это типы, которые FGES исключает как биологически перекрывающиеся (`CONTROLS_TO_DELETE`; для pan-macrophage — Monocytes и M1/M2), и считать их контролем значило бы штрафовать сигнатуру за правильную работу. Под этим отбором `Main4_Pan_macrophage_signature` — **1-я из 261** (gap = 10 031 при GOI 7866 против контроля −2166). Для сравнения, прежний ранговый композит `rank(метрика) + rank(−goi_cv)` держал её на 15–19 месте и в топ-10 не пускал ни на одной из семи метрик.

Прежний отбор по bootstrap-метрикам (`F1`, `Accuracy`, `Precision_score`, `Recall_score`, `Average_precision`, `ROC_AUC`, `PR_AUC` в композите с `goi_cv`) сохранён и пишет свои семь папок — с тем же новым оформлением.

**Набор 2 — специфичность internal.** Строки: только internal (BG) сигнатуры, все 19. Цвет — та же сырая медиана ssGSEA на общей шкале.

**Кроссвалидационный бэкфилл, `mode="merge"`.** Типы с нулём тестовых сэмплов (`Th17_cells`, `Eosinophils`, `Endothelium_lymph`) берутся целиком из кроссвалидации оригинальной когорты — объединять не с чем. Типы, где тестовые сэмплы есть, но их меньше 10 (`Th2_cells` 8, `Mast_cells` 8 после удаления train-сэмплов, `Monocytic_DC` 1), **объединяют тест и трейн** и скорятся на всех сэмплах сразу. Порог 10 передаётся в `backfill_rare_cell_types` явно; общий `DEFAULT_MIN_N = 20` в `crossval.py` не меняется. При пороге 10 `Plasmablasts` (12), `Keratinocytes` (15) и `Fibroblast_line` (10) трейном уже не добираются и остаются чистым held-out. Пометки: `*` — вся колонка из кроссвалидации, `†` — вся сигнатура оттуда же, `‡` — в колонке/строке есть объединённые тест+трейн данные.

Объединение на уровне скоров корректно потому, что ssGSEA — single-sample метод: скор сэмпла не зависит от состава когорты. Колонки берутся по пересечению, расхождение версий gene sets попадёт в лог, а не испортит медианы молча.

**Порядок строк** задаётся `fges_order_by_cell_type` — FGES сортируются по позиции своего GOI-типа на оси X.

In [ ]:
# ── Доп. фигуры: подготовка данных ───────────────────────────────────────
# Редкие типы клеток новой когорты (Th17_cells / Eosinophils / Endothelium_lymph
# = 0 семплов; Th2_cells = 8; Mast_cells = 8 после удаления train;
# Monocytic_DC = 1) добираются кроссвалидационными скорами оригинальной когорты.
#
# Порог здесь min_n=10 и передаётся явно: DEFAULT_MIN_N=20 в crossval.py общий
# для всех ноутбуков и остаётся как есть. При пороге 10 трейном НЕ добираются и
# остаются чистым held-out Plasmablasts (12), Keratinocytes (15) и
# Fibroblast_line (10) — при пороге 20 они попадали в merge.
#
# mode="merge": там, где тестовые сэмплы ЕСТЬ, но их мало, тест и трейн
# ОБЪЕДИНЯЮТСЯ и скор считается на всех сэмплах сразу (провенанс "merged", на
# фигурах помечается ‡). Там, где тестовых сэмплов нет вообще, объединять не с
# чем — такие типы остаются чисто кроссвалидационными ("cross_validation", †/*).
# Он же добавляет 4 редкие FGES, которых нет в 15-FGES scope этого ноутбука.
from signature_validation.benchmark.cohorts import MAP_RAW_RARE
from signature_validation.benchmark.crossval import (
    backfill_rare_cell_types,
    load_crossval_ssgseas,
)
from signature_validation.benchmark.heatmaps_supp import (
    CV_METRIC_KEY,
    HIGHER_IS_BETTER_METRICS,
    OTHER_CONTROLS_MEMBERS,
    SEPARATION_KEY,
    SUPP_COLUMN_ORDER,
    fges_order_by_cell_type,
    plot_internal_specificity_heatmap,
    plot_top_metric_heatmaps,
)

crossval_ssgseas = load_crossval_ssgseas(CROSSVAL_SSGSEAS_PATH)
mapping_ssgseas_supp, provenance_supp = backfill_rare_cell_types(
    mapping_ssgseas, crossval_ssgseas, public_cells_annot, min_n=10, mode="merge"
)

# FGES сортируются по позиции своего GOI-типа на оси X → диагональ читается.
FGES_ORDER_15 = fges_order_by_cell_type(MAP_RAW)                      # есть метрики
FGES_ORDER_19 = fges_order_by_cell_type({**MAP_RAW, **MAP_RAW_RARE})  # + 4 редкие

# Разбивка провенанса: сколько (FGES × group × cell_type) из какого источника.
prov_counts = pd.Series(
    [
        label
        for groups in provenance_supp.values()
        for labels in groups.values()
        for label in labels.values()
    ]
).value_counts()
logger.info("провенанс (FGES×group×cell_type): {}", prov_counts.to_dict())
logger.info(
    "supp heatmaps: {} FGES after backfill; {} named columns + Other controls ({} pooled types)",
    len(mapping_ssgseas_supp),
    len(SUPP_COLUMN_ORDER),
    len(OTHER_CONTROLS_MEMBERS),
)

In [ ]:
# ── Фигуры: ОДИН SVG на FGES, отдельная папка на каждый способ отбора ────
# Основной отбор — SEPARATION_KEY: "наиболее высокий в GOI, наиболее низкий в
# контроле". Строки ранжируются по РАЗРЫВУ МЕДИАН между GOI-семплами и ПУЛОМ
# контролей той же FGES: gap = median(GOI) − median(Control). Именно gap, потому
# что это то разделение, которое видно на фигуре — ячейки покрашены сырыми
# медианами, значит самый широкий разрыв это самый широкий скачок цвета.
# Cohen's d считается рядом и лежит в таблице колонкой d, но не сортирует: он
# делит на pooled SD и опускает сигнатуру за разброс внутри GOI, которого на
# фигуре не видно. Обратная сторона gap: суб-сигнатуры одной FGES живут на
# разных шкалах ssGSEA, поэтому крупный gene set может выиграть по разрыву, не
# разделяя когорты лучше в относительных единицах — на этот случай и колонка d.
#
# Считается прямо по тому ssGSEA, который фигура и рисует, поэтому (а) учтён
# кроссвалидационный бэкфилл (fges_metrics про него не знает), (б) один и тот же
# MSigDb-сет под двумя FGES скорится отдельно под каждой.
#
# Deleted_controls в контроль НЕ входят: это типы, которые FGES исключает как
# биологически перекрывающиеся (для pan-macrophage — Monocytes и M1/M2), и
# считать их контролем значило бы штрафовать сигнатуру за правильную работу.
#
# Цвет — СЫРАЯ медиана ssGSEA (без z-score), пределы берутся по ПОЛНОЙ матрице
# всех FGES и только потом нарезаются по файлам: иначе цвета двух файлов
# несопоставимы. Цифр в ячейках нет — их значение несёт colorbar. Шкала
# линейная; color_scale="symlog" доступен, но по умолчанию выключен: медианы
# занимают меньше одной декады (99.5% ячеек в 10^3–10^4), и log10 сжал бы
# полезный контраст (Macrophages 8688 vs NK 2955 → 3.94 vs 3.47) в 1/12 полосы.
PER_FGES_DIR = OUTPUT_DIR / "heatmaps_per_fges"

medians_sep, ranking_sep = plot_top_metric_heatmaps(
    mapping_ssgseas=mapping_ssgseas_supp,
    fges_metrics=fges_metrics,  # при отборе по separation не используется
    msigdb_gmt=msigdb_gmt,
    fges_order=FGES_ORDER_15,
    save_dir=PER_FGES_DIR,
    metric=SEPARATION_KEY,
    top_k=10,
    provenance=provenance_supp,
)
ranking_sep.to_csv(
    TABLES_DIR / "top10_separation_ranking_new_cohort.tsv", sep="\t", index=False
)
medians_sep.to_csv(TABLES_DIR / "top10_separation_medians_new_cohort.tsv", sep="\t")

# Сколько internal (BG) сигнатур попали в собственный топ-10 — прямая проверка
# того, что отбор поднимает "родную" сигнатуру FGES наверх.
internal_sep = ranking_sep[ranking_sep["is_internal"]]
logger.info(
    "separation: internal сигнатур в своём топ-10 — {}/{}",
    len(internal_sep),
    len(FGES_ORDER_15),
)

# Прежний отбор по bootstrap-метрикам сохранён для supplement: те же семь папок,
# но уже с квадратными ячейками, без аннотации и с сырыми медианами в цвете.
metric_rankings = {}
for metric in HIGHER_IS_BETTER_METRICS:
    medians_m, ranking_m = plot_top_metric_heatmaps(
        mapping_ssgseas=mapping_ssgseas_supp,
        fges_metrics=fges_metrics,
        msigdb_gmt=msigdb_gmt,
        fges_order=FGES_ORDER_15,
        save_dir=PER_FGES_DIR,
        metric=metric,
        cv_key=CV_METRIC_KEY,
        top_k=10,
        provenance=provenance_supp,
    )
    ranking_m.to_csv(
        TABLES_DIR / f"top10_{metric}_ranking_new_cohort.tsv", sep="\t", index=False
    )
    medians_m.to_csv(TABLES_DIR / f"top10_{metric}_medians_new_cohort.tsv", sep="\t")
    metric_rankings[metric] = ranking_m

logger.info(
    "готово: separation + {} метрик × {} FGES → {}",
    len(metric_rankings),
    ranking_sep["FGES"].nunique(),
    PER_FGES_DIR,
)
internal_sep[
    ["FGES", "Rank_in_fges", "Pool", "gap", "d", "goi_median", "control_median"]
]

In [ ]:
# ── Фигура 2: специфичность internal (BG) сигнатур ───────────────────────
# Показываем, по каким популяциям расходится НЕшкалированный ssGSEA одной
# сигнатуры. Цвет — сырая медиана на общей шкале, цифры в ячейках убраны:
# их значение читается с colorbar.
INTERNAL_HEATMAP_PATH = (
    OUTPUT_DIR / "heatmap_internal_specificity_raw_median_new_cohort.svg"
)

medians_internal = plot_internal_specificity_heatmap(
    mapping_ssgseas=mapping_ssgseas_supp,
    fges_order=FGES_ORDER_19,
    save_path=INTERNAL_HEATMAP_PATH,
    colorbar_path=INTERNAL_HEATMAP_PATH.with_name(
        INTERNAL_HEATMAP_PATH.stem + "_colorbar.svg"
    ),
    provenance=provenance_supp,
)

medians_internal.to_csv(
    TABLES_DIR / "internal_specificity_medians_new_cohort.tsv", sep="\t"
)
medians_internal.round(0)